# 01 — Corpus Construction & Benchmark Loading

Builds/loads the LegalRAG corpus and the 242-question benchmark, and reports
composition statistics (paper Tables 3–4). Run this first.


In [ ]:
# Configuration is centralised in src/config.py (single source of truth).
# If running outside the package, the constants below mirror that file.
import sys, os
sys.path.append(os.path.abspath(".."))
try:
    from src.config import *
    print("Loaded config from src.config")
except Exception as e:
    print("src.config not importable here; using inline constants.", e)
    RANDOM_SEED = 42
    RRF_KAPPA = 60
    DENSE_BATCH_SIZE = 128
    MAX_SEQ_LENGTH = 512
    NAIVE_K, STATIC_K = 5, 10
    ADAPTIVE_K = {"simple":5,"moderate":10,"complex":15}


## Setup, cleaning, and ID normalisation

In [ ]:
#@title 0.1 Install dependencies
!pip install -q datasets rank-bm25 sentence-transformers faiss-cpu pandas numpy matplotlib openpyxl tqdm scikit-learn scipy openai

In [ ]:
#@title 0.2 Imports and global configuration
import os, re, json, time, shutil, zipfile, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
warnings.filterwarnings('ignore')

BENCHMARK_CSV = '/content/legalrag_qa_benchmark.csv'
HF_CORPUS_ID  = 'Arailym-tleubayeva/LegalRAG'
OUTPUT_DIR    = Path('/content/legalrag_outputs')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 256
RRF_K = 60
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Output dir: {OUTPUT_DIR}')


def norm_id(v):
    """Normalize ids read from CSV/HF datasets, especially float-like ids such as 123.0."""
    if pd.isna(v):
        return ''
    s = str(v).strip()
    if s.lower() in {'', 'nan', 'none', 'null'}:
        return ''
    return re.sub(r'\.0$', '', s)


def clean_str(v):
    if pd.isna(v):
        return ''
    return str(v).strip()


def save_table(df: pd.DataFrame, name: str):
    """Save a result table as CSV and XLSX under OUTPUT_DIR."""
    csv_path = OUTPUT_DIR / f'{name}.csv'
    xlsx_path = OUTPUT_DIR / f'{name}.xlsx'
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    df.to_excel(xlsx_path, index=False)
    print(f'Saved: {csv_path.name}, {xlsx_path.name}')


def save_json(obj, name: str):
    path = OUTPUT_DIR / f'{name}.json'
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    print(f'Saved: {path.name}')

In [ ]:
#@title 0.3 clean_str with float/ID normalization (FIX for 788.0 vs "788" bug)
import math

def clean_str(x):
    """Normalize any value to a clean string ID.
    Critical fix: pandas reads numeric chunk IDs as float (788.0), while corpus
    chunk IDs are strings ('788'). Without normalization, gold==retrieved checks
    silently fail. This maps 788.0 / '788.0' / 788 / '788' all to '788'."""
    if x is None:
        return ''
    if isinstance(x, float):
        if math.isnan(x):
            return ''
        if x.is_integer():
            return str(int(x))
        return str(x).strip()
    if isinstance(x, int):
        return str(x)
    s = str(x).strip()
    if s.lower() in ('nan', 'none', ''):
        return ''
    if s.endswith('.0') and s[:-2].isdigit():
        s = s[:-2]
    return s

assert clean_str(788.0) == clean_str('788') == clean_str(788) == clean_str('788.0') == '788', 'clean_str fix failed'
print('clean_str fixed: 788.0 / "788.0" / 788 / "788" all ->', clean_str(788.0))

In [ ]:
#@title 0.4 Normalize gold ID columns at load time (root-cause fix)
ID_COLS = ['gold_chunk_id', 'gold_doc_id', 'gold_chunk_id_str', 'gold_doc_id_str',
           'gold_source', 'gold_source_str']

def _normalize_id_columns(df):
    for col in ID_COLS:
        if col in df.columns:
            df[col] = (df[col].astype('string')
                       .str.replace(r'\.0$', '', regex=True)
                       .fillna(''))
    return df

for _name in ['bench_all', 'bench_ans', 'bench_unans']:
    if _name in dir():
        globals()[_name] = _normalize_id_columns(globals()[_name].copy())
        print(f'Normalized ID columns in {_name} ({len(globals()[_name])} rows)')

for _name in ['bench_all', 'bench_ans', 'bench_unans']:
    if _name in dir():
        _df = globals()[_name]
        if 'gold_chunk_id_str' not in _df.columns and 'gold_chunk_id' in _df.columns:
            _df['gold_chunk_id_str'] = _df['gold_chunk_id'].map(clean_str)
        if 'gold_doc_id_str' not in _df.columns and 'gold_doc_id' in _df.columns:
            _df['gold_doc_id_str'] = _df['gold_doc_id'].map(clean_str)
        if 'gold_source_str' not in _df.columns and 'gold_source' in _df.columns:
            _df['gold_source_str'] = _df['gold_source'].map(clean_str)

print('Gold ID columns normalized and *_str helpers ensured.')

In [ ]:
#@title 0.5 Verify the ID fix on one question (must show in pool: True)
_q = bench_ans.iloc[0]['question']
_cands = rrf([bm25_ret(bm25_text, _q, 20), dense_ret(faiss_index, query_embs_ans[0], 20)], k=20)
_gold = clean_str(bench_ans.iloc[0].get('gold_chunk_id_str', bench_ans.iloc[0].get('gold_chunk_id', '')))
_ids = [clean_str(x) for x in _cands['retrieved_ids']]
print('gold:', _gold)
print('first retrieved ids:', _ids[:5])
print('gold in pool:', _gold in _ids, '| position:', (_ids.index(_gold) if _gold in _ids else 'NOT FOUND'))
print('--> If "gold in pool: True", the ID type bug is fixed. Now re-run all retrieval cells.')

## Load benchmark and corpus

In [ ]:
#@title 1.1 Load `legalrag_qa_benchmark.csv`
from pathlib import Path

# Colab upload fallback
if not Path(BENCHMARK_CSV).exists():
    local_fallbacks = [
        '/mnt/data/legalrag_qa_benchmark(2).csv',
        '/mnt/data/legalrag_qa_benchmark.csv',
        'legalrag_qa_benchmark.csv',
    ]
    found = next((p for p in local_fallbacks if Path(p).exists()), None)
    if found:
        BENCHMARK_CSV = found
        print(f'Using local fallback: {BENCHMARK_CSV}')
    else:
        try:
            from google.colab import files as cf
            uploaded = cf.upload()
            for fname in uploaded:
                Path(fname).rename('/content/legalrag_qa_benchmark.csv')
            BENCHMARK_CSV = '/content/legalrag_qa_benchmark.csv'
        except Exception as e:
            raise FileNotFoundError('Benchmark CSV not found. Upload legalrag_qa_benchmark.csv.') from e

bench_all = pd.read_csv(BENCHMARK_CSV, encoding='utf-8-sig')
bench_all = bench_all.reset_index(drop=False).rename(columns={'index': 'row_id'})

required_cols = [
    'question_id', 'question', 'gold_answer', 'gold_evidence', 'gold_chunk_id',
    'gold_doc_id', 'gold_source', 'gold_article', 'gold_paragraph', 'language',
    'answerability', 'legal_domain', 'complexity', 'question_type'
]
missing = [c for c in required_cols if c not in bench_all.columns]
if missing:
    raise ValueError(f'Missing benchmark columns: {missing}')

for col in ['gold_chunk_id', 'gold_doc_id', 'gold_source', 'gold_article', 'gold_paragraph']:
    bench_all[f'{col}_str'] = bench_all[col].apply(norm_id if 'id' in col else clean_str)

bench_all['answerability'] = bench_all['answerability'].fillna('').astype(str).str.strip()
bench_all['complexity'] = bench_all['complexity'].fillna('').astype(str).str.strip()
bench_all['question_type'] = bench_all['question_type'].fillna('').astype(str).str.strip()
bench_all['legal_domain'] = bench_all['legal_domain'].fillna('').astype(str).str.strip()
bench_all['question'] = bench_all['question'].fillna('').astype(str)

bench_ans = bench_all[bench_all['answerability'].eq('answerable')].reset_index(drop=True)
bench_unans = bench_all[bench_all['answerability'].eq('unanswerable')].reset_index(drop=True)

print(f'Benchmark rows: {len(bench_all)}')
print(f'Answerable: {len(bench_ans)} | Unanswerable: {len(bench_unans)}')
print(f'Unique questions: {bench_all["question"].nunique()}')
print('Answerable complexity:', dict(bench_ans['complexity'].value_counts()))

In [ ]:
#@title 1.2 Experiment 0 — Benchmark Sanity Analysis
sanity_rows = [
    ('Total questions', len(bench_all)),
    ('Answerable', int(bench_all['answerability'].eq('answerable').sum())),
    ('Unanswerable', int(bench_all['answerability'].eq('unanswerable').sum())),
    ('Simple answerable', int(bench_ans['complexity'].eq('simple').sum())),
    ('Moderate answerable', int(bench_ans['complexity'].eq('moderate').sum())),
    ('Complex answerable', int(bench_ans['complexity'].eq('complex').sum())),
    ('Deadline questions', int(bench_all['question_type'].eq('deadline').sum())),
    ('Definition questions', int(bench_all['question_type'].eq('definition').sum())),
    ('Fact lookup questions', int(bench_all['question_type'].eq('fact_lookup').sum())),
    ('Unique questions', int(bench_all['question'].nunique())),
    ('Article-annotated', int(bench_all['gold_article_str'].astype(bool).sum())),
    ('Paragraph-annotated', int(bench_all['gold_paragraph_str'].astype(bool).sum())),
    ('Language', ', '.join(sorted(bench_all['language'].dropna().astype(str).unique()))),
]
sanity_df = pd.DataFrame(sanity_rows, columns=['Statistic', 'Value'])
display(sanity_df)
save_table(sanity_df, 'table00_benchmark_sanity')

answerability_dist = bench_all['answerability'].value_counts(dropna=False).rename_axis('answerability').reset_index(name='count')
complexity_dist = bench_ans['complexity'].value_counts(dropna=False).rename_axis('complexity').reset_index(name='count')
qtype_dist = bench_all['question_type'].value_counts(dropna=False).rename_axis('question_type').reset_index(name='count')
domain_dist = bench_all['legal_domain'].value_counts(dropna=False).rename_axis('legal_domain').reset_index(name='count')

save_table(answerability_dist, 'table00_answerability_distribution')
save_table(complexity_dist, 'table00_answerable_complexity_distribution')
save_table(qtype_dist, 'table00_question_type_distribution')
save_table(domain_dist, 'table00_legal_domain_distribution')

LIMITATION_TEXT = (
    'The benchmark is dominated by deadline and definition questions. '
    'Multi-hop, procedure, eligibility, obligation, and exception-based question types are absent '
    'from the current benchmark and are reserved for future work.'
)
print('\nLimitation to report in the paper:')
print(LIMITATION_TEXT)

In [ ]:
#@title 1.3 Dataset composition figures
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False})

fig, ax = plt.subplots(figsize=(5, 5))
answerability_dist.set_index('answerability')['count'].plot(kind='pie', autopct='%1.1f%%', ax=ax)
ax.set_ylabel('')
ax.set_title('Benchmark Answerability')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig00_answerability.png', dpi=180, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
complexity_dist.set_index('complexity')['count'].reindex(['simple', 'moderate', 'complex']).dropna().plot(kind='bar', ax=ax)
ax.set_title('Answerable Questions by Complexity')
ax.set_xlabel('Complexity')
ax.set_ylabel('Count')
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig00_complexity.png', dpi=180, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
qtype_dist.set_index('question_type')['count'].plot(kind='bar', ax=ax)
ax.set_title('Question Type Distribution')
ax.set_xlabel('Question type')
ax.set_ylabel('Count')
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig00_question_type.png', dpi=180, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
domain_dist.set_index('legal_domain')['count'].plot(kind='bar', ax=ax)
ax.set_title('Legal Domain Distribution')
ax.set_xlabel('Legal domain')
ax.set_ylabel('Count')
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig00_legal_domain.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
#@title 2.1 Load `Arailym-tleubayeva/LegalRAG`
from datasets import load_dataset

ds = load_dataset(HF_CORPUS_ID)
split = 'train' if 'train' in ds else list(ds.keys())[0]
corpus_df = ds[split].to_pandas().reset_index(drop=True)

print(f'HF dataset: {HF_CORPUS_ID} | split={split}')
print(f'Corpus rows: {len(corpus_df):,}')
print('Columns:', corpus_df.columns.tolist())


def choose_col(candidates, required=True):
    for c in candidates:
        if c in corpus_df.columns:
            return c
    if required:
        raise ValueError(f'None of these corpus columns found: {candidates}')
    return None

CHUNK_COL = choose_col(['chunk_id', 'id', 'chunkId'])
DOC_COL = choose_col(['doc_id', 'document_id', 'docid', 'documentId'], required=False)
SOURCE_COL = choose_col(['source', 'url', 'source_url', 'link'], required=False)
TEXT_COL = choose_col(['text', 'content', 'chunk_text'])
NORM_TEXT_COL = choose_col(['normalized_text', 'text_normalized', 'norm_text'], required=False)
TITLE_COL = choose_col(['title', 'doc_title', 'document_title'], required=False)

corpus_df['chunk_id_str'] = corpus_df[CHUNK_COL].apply(norm_id)
corpus_df['doc_id_str'] = corpus_df[DOC_COL].apply(norm_id) if DOC_COL else ''
corpus_df['source_str'] = corpus_df[SOURCE_COL].apply(clean_str) if SOURCE_COL else ''
corpus_df['text_str'] = corpus_df[TEXT_COL].fillna('').astype(str)
corpus_df['title_str'] = corpus_df[TITLE_COL].fillna('').astype(str) if TITLE_COL else ''

c_ids = corpus_df['chunk_id_str'].tolist()
c_docs = corpus_df['doc_id_str'].tolist()
c_srcs = corpus_df['source_str'].tolist()
c_text = corpus_df['text_str'].tolist()
c_titles = corpus_df['title_str'].tolist()

id_to_pos = {cid: i for i, cid in enumerate(c_ids)}
id2text = dict(zip(c_ids, c_text))
id2doc = dict(zip(c_ids, c_docs))
id2src = dict(zip(c_ids, c_srcs))
id2title = dict(zip(c_ids, c_titles))

gold_chunks = set(bench_ans['gold_chunk_id_str'].dropna().astype(str)) - {''}
covered = gold_chunks & set(c_ids)
coverage = len(covered) / max(1, len(gold_chunks))
print(f'Gold chunk coverage: {len(covered)}/{len(gold_chunks)} ({coverage:.1%})')

corpus_stats = pd.DataFrame([
    ('HF corpus id', HF_CORPUS_ID),
    ('Split', split),
    ('Rows / chunks', len(corpus_df)),
    ('Chunk id column', CHUNK_COL),
    ('Document id column', DOC_COL or ''),
    ('Source column', SOURCE_COL or ''),
    ('Text column', TEXT_COL),
    ('Normalized text column', NORM_TEXT_COL or 'not available'),
    ('Gold chunk coverage', f'{len(covered)}/{len(gold_chunks)} ({coverage:.1%})'),
], columns=['Statistic', 'Value'])
display(corpus_stats)
save_table(corpus_stats, 'table00_corpus_loading_stats')